# 02 — Embedding & Clustering

This notebook demonstrates:
1. Embedding audience segments with sentence-transformers
2. HDBSCAN clustering to find cross-platform groups
3. Validating that cross-platform segments land in the same clusters
4. UMAP visualization of the embedding space

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
from collections import Counter, defaultdict
import plotly.express as px

from data_loader import load_all
from embedder import load_model, embed_segments, build_faiss_index, embed_texts, search_index
from clustering import cluster_segments, build_audience_groups, assign_noise_to_nearest, print_cluster_summary

## Step 1: Load and Embed

In [ ]:
segments = load_all()
model = load_model()
embeddings = embed_segments(segments, model)
print(f"Embedding matrix: {embeddings.shape}")

## Step 2: Sanity Check — Similarity Search

In [ ]:
index = build_faiss_index(embeddings)

queries = ["luxury cars", "pet food", "fitness", "real estate", "basketball"]
for q in queries:
    q_emb = embed_texts([q], model)
    scores, indices = search_index(index, q_emb, top_k=5)
    print(f"\nQuery: '{q}'")
    for score, idx in zip(scores, indices):
        if idx < 0: continue
        s = segments[idx]
        print(f"  [{s.platform:10s}] {s.name:35s} (score: {score:.3f})")

## Step 3: HDBSCAN Clustering

In [ ]:
labels, clusterer = cluster_segments(embeddings, segments)
groups = build_audience_groups(labels, embeddings, segments)

# Assign noise to nearest cluster
labels = assign_noise_to_nearest(labels, embeddings, groups)
groups = build_audience_groups(labels, embeddings, segments)

print_cluster_summary(groups, segments)

## Step 4: Cross-Platform Validation

The key question: do segments from different platforms that target the same audience land in the same cluster?

In [ ]:
# Find clusters that span the most platforms
multi_plat = [(g, len(g.platforms)) for g in groups]
multi_plat.sort(key=lambda x: (-x[1], -x[0].member_count))

print("Clusters spanning ALL 6 platforms:")
id_to_seg = {s.id: s for s in segments}

for g, n_plat in multi_plat[:10]:
    if n_plat < 4: break
    print(f"\n[{g.name}] — {g.member_count} members, {n_plat} platforms")
    by_plat = defaultdict(list)
    for sid in g.segment_ids:
        seg = id_to_seg[sid]
        by_plat[seg.platform].append(seg.name)
    for plat in sorted(by_plat):
        names = by_plat[plat][:3]
        print(f"  {plat:12s}: {', '.join(names)}")

## Step 5: UMAP Visualization

In [ ]:
import umap

# Sample for speed
sample_size = 3000
indices_sample = np.random.choice(len(embeddings), sample_size, replace=False)
emb_sample = embeddings[indices_sample]

reducer = umap.UMAP(n_components=2, metric='cosine', random_state=42)
coords = reducer.fit_transform(emb_sample.astype(np.float64))

platforms = [segments[i].platform for i in indices_sample]
names = [segments[i].name for i in indices_sample]

fig = px.scatter(
    x=coords[:, 0], y=coords[:, 1],
    color=platforms,
    hover_name=names,
    title='Audience Segments in Embedding Space (UMAP)',
    labels={'x': 'UMAP-1', 'y': 'UMAP-2', 'color': 'Platform'},
    template='plotly_dark',
    width=1000, height=700,
)
fig.update_traces(marker=dict(size=4, opacity=0.7))
fig.show()